# Message Content Caching — Converse API

This notebook demonstrates how to cache message content (user-provided text) using the **Converse** and **ConverseStream** APIs. The `cachePoint` syntax shown here is model-agnostic — it works identically with Anthropic Claude, Amazon Nova, and any future Bedrock model that supports prompt caching.

## What you'll learn

- **Converse API**: `cachePoint` as a standalone content block (simplified + multiple checkpoints)
- **ConverseStream API**: Same `cachePoint` syntax with streaming response parsing
- **Streaming TTFT benchmark**: Measure latency improvement from cache hits

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `boto3 >= 1.35.76`, `matplotlib`, `pandas`

## Token threshold

Claude Sonnet 4.6 requires at least **1,024 tokens** per cache checkpoint.

In [ ]:
!pip install --upgrade --quiet boto3 matplotlib pandas

In [ ]:
import boto3
import json
import time
import pandas as pd
import matplotlib.pyplot as plt

MODEL_ID = "global.anthropic.claude-sonnet-4-6-v1:0"
AWS_REGION = "us-west-2"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"boto3 version: {boto3.__version__}")
print(f"Model: {MODEL_ID}")
print(f"Region: {AWS_REGION}")

## Sample Content

Space-themed text split into 4 sections. Each section exceeds the 1,024-token minimum for Claude Sonnet 4.6.

In [ ]:
SPACE_SECTION_1 = """
The universe is a vast and mysterious expanse that has captivated human imagination for millennia. From the earliest civilizations who looked up at the night sky and wondered about the nature of the stars, to modern astronomers using sophisticated telescopes and spacecraft to explore distant galaxies, our quest to understand the cosmos continues unabated.

Our solar system, located in the Milky Way galaxy, is home to eight planets, numerous dwarf planets, and countless smaller objects including asteroids, comets, and meteoroids. The Sun, a middle-aged G-type main-sequence star, provides the energy that sustains life on Earth and influences the dynamics of all objects within its gravitational reach.

Mercury, the innermost planet, experiences extreme temperature variations due to its proximity to the Sun and lack of substantial atmosphere. Venus, often called Earth's twin due to its similar size, has a thick atmosphere composed primarily of carbon dioxide, creating a runaway greenhouse effect that makes it the hottest planet in our solar system. Earth, our home, is the only known planet to harbor life, with its unique combination of liquid water, moderate temperatures, and protective magnetic field.

Mars, the Red Planet, has long been a subject of fascination and speculation about the possibility of extraterrestrial life. Its rusty appearance comes from iron oxide prevalent on its surface. The planet features the largest volcano in the solar system, Olympus Mons, and a canyon system, Valles Marineris, that dwarfs the Grand Canyon. Recent Mars missions have discovered evidence of ancient river systems and the presence of water ice beneath the surface.

The asteroid belt, located between Mars and Jupiter, contains millions of rocky objects ranging from small boulders to the dwarf planet Ceres. These remnants from the early solar system provide valuable insights into planetary formation and the conditions that existed billions of years ago. Scientists study these asteroids to understand the building blocks of planets and the early solar system's chemical composition. Some asteroids contain valuable metals and minerals that may one day be mined for space-based industries.

The terrestrial planets share common characteristics: rocky compositions, relatively small sizes compared to gas giants, and solid surfaces. Mercury's heavily cratered surface resembles our Moon, preserving a record of impacts from the early solar system. Venus's dense atmosphere traps heat so effectively that its surface temperature exceeds that of Mercury, despite being farther from the Sun. Earth's plate tectonics continuously reshape its surface, while Mars shows evidence of past geological activity including ancient volcanoes and water-carved channels.
"""

SPACE_SECTION_2 = """
Jupiter, the largest planet, is a gas giant composed primarily of hydrogen and helium. Its Great Red Spot, a persistent anticyclonic storm, has been observed for over 400 years. Jupiter's intense magnetic field and numerous moons, including the four Galilean satellites discovered by Galileo Galilei in 1610, make it a miniature solar system in its own right. Europa, one of these moons, is believed to have a subsurface ocean that could potentially harbor life.

Saturn, famous for its spectacular ring system, is another gas giant with dozens of moons. Titan, its largest moon, has a thick atmosphere and liquid hydrocarbon lakes, making it one of the most intriguing bodies in the solar system for astrobiological research. The Cassini-Huygens mission provided unprecedented details about Saturn and its moons during its 13-year exploration.

Uranus and Neptune, the ice giants, reside in the outer reaches of our solar system. Uranus rotates on its side, likely due to a massive impact early in its history. Neptune, the windiest planet, features storms with wind speeds exceeding 2,000 kilometers per hour. Both planets have ring systems, though less prominent than Saturn's.

Beyond Neptune lies the Kuiper Belt, a region populated by icy bodies including the dwarf planet Pluto. The New Horizons mission's flyby of Pluto in 2015 revealed a geologically active world with nitrogen glaciers and a hazy atmosphere. Even further out is the Oort Cloud, a hypothetical spherical shell of icy objects that may extend halfway to the nearest star.

The outer solar system remains largely unexplored compared to the inner planets. Only Voyager 2 has visited both Uranus and Neptune, conducting brief flybys in the 1980s. Future missions are being planned to study these ice giants in more detail, potentially including orbiters and atmospheric probes. The moons of the outer planets present exciting targets for astrobiology, with Europa, Enceladus, and Titan all showing signs of environments that could support life.

Gas giants and ice giants differ fundamentally in composition. While Jupiter and Saturn are primarily hydrogen and helium, Uranus and Neptune contain significant amounts of water, ammonia, and methane ices. This distinction gives the ice giants their characteristic blue-green colors and different internal structures compared to their larger neighbors.
"""

SPACE_SECTION_3 = """
Exoplanet research has revolutionized our understanding of planetary systems. The Kepler space telescope discovered thousands of planets orbiting other stars, revealing that planets are common throughout our galaxy. Some of these exoplanets reside in the habitable zone of their stars, where liquid water could exist on the surface.

The search for extraterrestrial intelligence, known as SETI, uses radio telescopes to listen for signals from advanced civilizations. While no definitive signals have been detected, the Drake Equation provides a framework for estimating the number of communicating civilizations in our galaxy.

Black holes, regions of spacetime where gravity is so strong that nothing can escape, represent some of the most extreme objects in the universe. Stellar black holes form from the collapse of massive stars, while supermassive black holes, containing millions to billions of solar masses, reside at the centers of most galaxies including our own.

The Event Horizon Telescope collaboration captured the first image of a black hole's shadow in 2019, confirming predictions from Einstein's general theory of relativity. Gravitational wave detectors have also observed the mergers of black holes and neutron stars, opening a new window on the universe.

Dark matter and dark energy together comprise about 95% of the universe's total mass-energy content. Dark matter, which does not emit or absorb light, reveals its presence through gravitational effects on visible matter. Dark energy, even more mysterious, appears to be driving the accelerated expansion of the universe.

The cosmic microwave background radiation, discovered in 1965, provides a snapshot of the universe approximately 380,000 years after the Big Bang. Detailed measurements of this radiation have confirmed the Big Bang theory and revealed information about the early universe's composition and geometry.

Modern telescopes have detected exoplanets using multiple methods including transit photometry, radial velocity measurements, and direct imaging. Transit photometry measures the slight dimming of starlight when a planet passes in front of its host star. Radial velocity detects the wobble in a star's motion caused by orbiting planets. Direct imaging captures actual light from exoplanets, though this remains challenging due to the overwhelming brightness of host stars.
"""

SPACE_SECTION_4 = """
Stellar evolution describes how stars change over their lifetimes. Stars form in molecular clouds when gravity causes dense regions to collapse. Nuclear fusion in the core converts hydrogen to helium, releasing enormous amounts of energy. When stars exhaust their nuclear fuel, their fate depends on their mass: smaller stars become white dwarfs, medium stars may become neutron stars, and the most massive stars explode as supernovae, potentially leaving behind black holes.

Galaxies, containing billions of stars, come in various shapes including spiral, elliptical, and irregular. The Milky Way is a barred spiral galaxy approximately 100,000 light-years in diameter. Galaxies often cluster together, forming groups and superclusters connected by cosmic filaments of dark matter and gas.

Space exploration has achieved remarkable milestones since the launch of Sputnik in 1957. Human spaceflight began with Yuri Gagarin's orbit in 1961 and culminated in the Apollo Moon landings. The International Space Station has hosted continuous human presence in space since 2000. Future missions aim to return humans to the Moon and eventually send astronauts to Mars.

Private space companies have transformed the industry, developing reusable rockets that have dramatically reduced launch costs. These advances have enabled new possibilities for satellite constellations, space tourism, and lunar exploration. Plans for permanent lunar bases and Mars colonies represent the next frontier of human expansion into space.

The study of astrobiology examines the origin, evolution, and distribution of life in the universe. Scientists search for biosignatures in planetary atmospheres and analyze extremophiles on Earth to understand the limits of life. The discovery of organic molecules on Mars and in the plumes of Enceladus fuels speculation about the possibility of life elsewhere in our solar system.

Space telescopes have revolutionized astronomy by observing wavelengths blocked by Earth's atmosphere. The Hubble Space Telescope has provided stunning images and groundbreaking discoveries for over three decades. The James Webb Space Telescope, launched in 2021, observes in infrared to study the earliest galaxies and probe planetary atmospheres for signs of life.

Understanding the universe requires collaboration across disciplines including physics, chemistry, biology, and engineering. International cooperation has enabled ambitious projects like the International Space Station and large-scale observatories. The quest to explore space continues to inspire new generations of scientists and engineers who will push the boundaries of human knowledge and capability.
"""

SPACE_CONTEXT = SPACE_SECTION_1 + SPACE_SECTION_2 + SPACE_SECTION_3 + SPACE_SECTION_4
QUESTION = "Based on the context above about space, what are the two ice giant planets in our solar system and what makes them unique?"

## 1. Converse API — Simplified (Single Checkpoint)

Place a `cachePoint` block after all static content. Bedrock scans backward to find the longest cacheable prefix.

```python
content = [
    {"text": "<static content>"},
    {"cachePoint": {"type": "default"}},  # checkpoint
    {"text": "<user question>"}            # dynamic
]
```

In [ ]:
def converse_simplified():
    content = [
        {"text": SPACE_CONTEXT},
        {"cachePoint": {"type": "default"}},
        {"text": QUESTION}
    ]

    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )
    return response["usage"]

# Request 1 — cache write
usage1 = converse_simplified()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_simplified()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## 2. Converse API — Multiple Checkpoints

Place a `cachePoint` after each section for granular control.

In [ ]:
def converse_multiple():
    content = [
        {"text": SPACE_SECTION_1},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_2},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_3},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_4},
        {"cachePoint": {"type": "default"}},
        {"text": QUESTION}
    ]

    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )
    return response["usage"]

# Request 1 — cache write
usage1 = converse_multiple()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_multiple()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## 3. ConverseStream API — Simplified (Single Checkpoint)

Same `cachePoint` syntax, but with streaming output. Cache metrics arrive in the `metadata` event at the end of the stream:

```python
for event in response["stream"]:
    if "metadata" in event:
        usage = event["metadata"]["usage"]
        # usage["cacheWriteInputTokens"], usage["cacheReadInputTokens"]
```

In [ ]:
def converse_stream_simplified():
    content = [
        {"text": SPACE_CONTEXT},
        {"cachePoint": {"type": "default"}},
        {"text": QUESTION}
    ]

    response = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )

    text = ""
    usage = {}
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            text += event["contentBlockDelta"]["delta"].get("text", "")
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})

    return usage

# Request 1 — cache write
usage1 = converse_stream_simplified()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_stream_simplified()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## 4. ConverseStream API — Multiple Checkpoints

Same multiple checkpoint pattern with streaming.

In [ ]:
def converse_stream_multiple():
    content = [
        {"text": SPACE_SECTION_1},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_2},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_3},
        {"cachePoint": {"type": "default"}},
        {"text": SPACE_SECTION_4},
        {"cachePoint": {"type": "default"}},
        {"text": QUESTION}
    ]

    response = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )

    text = ""
    usage = {}
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            text += event["contentBlockDelta"]["delta"].get("text", "")
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})

    return usage

# Request 1 — cache write
usage1 = converse_stream_multiple()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_stream_multiple()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## Streaming TTFT Benchmark

Measure Time-To-First-Token using ConverseStream with and without caching.

In [ ]:
def measure_ttft(use_cache=True, iterations=5):
    """Measure TTFT using ConverseStream."""
    ttfts = []

    for i in range(iterations):
        if use_cache:
            content = [
                {"text": SPACE_CONTEXT},
                {"cachePoint": {"type": "default"}},
                {"text": QUESTION}
            ]
        else:
            content = [{"text": SPACE_CONTEXT + "\n\n" + QUESTION}]

        start = time.time()
        response = bedrock.converse_stream(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": content}],
            inferenceConfig={"maxTokens": 512}
        )

        for event in response["stream"]:
            if "contentBlockDelta" in event:
                ttft = time.time() - start
                ttfts.append(ttft)
                for _ in response["stream"]:
                    pass
                break

        time.sleep(0.5)

    return ttfts

print("Measuring TTFT with caching (5 iterations)...")
ttfts_cached = measure_ttft(use_cache=True, iterations=5)

time.sleep(2)

print("Measuring TTFT without caching (5 iterations)...")
ttfts_uncached = measure_ttft(use_cache=False, iterations=5)

print(f"\nCached TTFT   — median: {sorted(ttfts_cached)[len(ttfts_cached)//2]:.3f}s")
print(f"Uncached TTFT — median: {sorted(ttfts_uncached)[len(ttfts_uncached)//2]:.3f}s")

In [ ]:
df = pd.DataFrame({
    "TTFT (s)": ttfts_cached + ttfts_uncached,
    "Mode": ["Cached"] * len(ttfts_cached) + ["Uncached"] * len(ttfts_uncached)
})

fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column="TTFT (s)", by="Mode", ax=ax)
ax.set_title("Streaming TTFT: Cached vs Uncached")
ax.set_ylabel("Time to First Token (seconds)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated message content caching using the Converse and ConverseStream APIs:

- The `cachePoint` syntax is **model-agnostic** — works with Claude, Nova, and any supported model
- **Simplified mode**: single checkpoint covers all static content
- **Multiple checkpoints**: granular caching of individual sections
- The first request writes to cache (`cacheWriteInputTokens > 0`), subsequent requests read from cache (`cacheReadInputTokens > 0`)
- Streaming TTFT is measurably lower on cache hits

For Anthropic-specific InvokeModel syntax (`cache_control`), see [invoke_model_api/anthropic/](../../invoke_model_api/anthropic/).

For more details, see the [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).